# 04. Shadow-model attack (MIA), simple attacks and the hidden-attribute test (v2)
Uses the models saved by notebook 02. Members are the target **training** patients, non-members are the target **test** patients. Normal CPU runtime.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('adversarial-robustness-toolbox')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': None, 'art': '1.20.1', 'sklearn': '1.6.1'}


In [ ]:
from art.attacks.inference.membership_inference import MembershipInferenceBlackBox
from art.estimators.classification import PyTorchClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_score, recall_score

Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
Xs_tr, ys_tr = load_split("shadow_train"); Xs_te, ys_te = load_split("shadow_test")
input_dim = Xtr.shape[1]; NAMES = model_names()
out_csv = RESULTS + "multiseed_results.csv"
done = set(pd.read_csv(out_csv)["seed"]) if os.path.exists(out_csv) else set()

def build_attack(seed):
    shadow = train_plain(Xs_tr, ys_tr, EPOCHS, 1000 + seed)[0]            # same recipe as the 50-epoch target
    xi, yi, xo, yo = balanced(Xs_tr, ys_tr, Xs_te, ys_te, np.random.RandomState(seed))
    art_m = PyTorchClassifier(model=shadow, loss=nn.CrossEntropyLoss(), input_shape=(input_dim,), nb_classes=2, device_type="cpu")
    np.random.seed(seed)
    atk = MembershipInferenceBlackBox(art_m, attack_model_type="rf")
    atk.fit(pred=proba(shadow, xi), y=yi, test_pred=proba(shadow, xo), test_y=yo)
    return atk

for seed in SEEDS:
    if seed in done:
        continue
    atk = build_attack(seed)
    n = len(Xte); idx = np.random.RandomState(seed).choice(len(Xtr), n, replace=False)
    labels = np.r_[np.ones(n), np.zeros(n)]
    rows = []
    for name in NAMES:
        m = load_model(name, seed, input_dim)
        ptr, pte = proba(m, Xtr), proba(m, Xte)
        s_in = atk.infer(x=None, y=ytr[idx], pred=ptr[idx], probabilities=True).ravel()
        s_out = atk.infer(x=None, y=yte, pred=pte, probabilities=True).ravel()
        sc = np.r_[s_in, s_out]; guess = (sc > 0.5).astype(int)
        meta = json.load(open(f"{FINAL}{name}_seed{seed}.json"))
        rows.append({"seed": seed, "model": name, "eps": meta.get("target_epsilon"), "spent_eps": meta.get("spent_epsilon"),
                     "noise": meta.get("noise_multiplier"), "train_acc": float((ptr.argmax(1) == ytr).mean()),
                     "test_acc": float((pte.argmax(1) == yte).mean()), "test_auc": roc_auc_score(yte, pte[:, 1]),
                     "test_pr_auc": average_precision_score(yte, pte[:, 1]), "test_brier": brier_score_loss(yte, pte[:, 1]),
                     "attack_auc": roc_auc_score(labels, sc), "attack_acc": float((guess == labels).mean()),
                     "precision": precision_score(labels, guess, zero_division=0), "recall": recall_score(labels, guess, zero_division=0)})
    pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
    print("seed", seed, "done")

res = pd.read_csv(out_csv)
print(res.groupby("model")[["test_auc", "test_pr_auc", "attack_auc", "test_acc"]].agg(["mean", "std"]).reindex(NAMES).round(3).to_string())


seed 1 done
seed 2 done
seed 3 done
seed 4 done
seed 5 done
               test_auc        test_pr_auc        attack_auc        test_acc       
                   mean    std        mean    std       mean    std     mean    std
model                                                                              
baseline_e50      0.542  0.028       0.129  0.010      0.607  0.008    0.830  0.009
baseline_early    0.609  0.003       0.165  0.002      0.500  0.005    0.889  0.000
dpsgd_eps0.5      0.567  0.008       0.142  0.005      0.498  0.006    0.889  0.000
dpsgd_eps1        0.582  0.007       0.148  0.006      0.499  0.006    0.889  0.000
dpsgd_eps2        0.592  0.005       0.152  0.005      0.502  0.011    0.889  0.000
dpsgd_eps5        0.600  0.003       0.155  0.003      0.495  0.005    0.889  0.000
dpsgd_eps10       0.604  0.003       0.157  0.003      0.499  0.009    0.889  0.000
dpsgd_eps100      0.609  0.003       0.161  0.003      0.502  0.005    0.889  0.000


In [ ]:
# Simple attacks: loss, confidence, label-only. AUC and the catch rate at low false alarms.
labels = np.r_[np.ones(len(ytr)), np.zeros(len(yte))]
rows = []
for seed in SEEDS:
    for name in NAMES:
        m = load_model(name, seed, input_dim)
        s_in = simple_attack_scores(get_logits(m, Xtr), ytr); s_out = simple_attack_scores(get_logits(m, Xte), yte)
        for atk in ("loss", "confidence", "label_only"):
            sc = np.r_[s_in[atk], s_out[atk]]
            rows.append({"seed": seed, "model": name, "attack": atk, "auc": roc_auc_score(labels, sc),
                         "tpr@1%fpr": tpr_at_fpr(labels, sc, 0.01), "tpr@0.1%fpr": tpr_at_fpr(labels, sc, 0.001)})
sa = pd.DataFrame(rows); sa.to_csv(RESULTS + "simple_attacks.csv", index=False)
for metric in ("auc", "tpr@1%fpr"):
    print(metric); print(sa.pivot_table(index="model", columns="attack", values=metric).reindex(NAMES).round(3).to_string())


auc
attack          confidence  label_only   loss
model                                        
baseline_e50         0.513       0.580  0.570
baseline_early       0.479       0.499  0.492
dpsgd_eps0.5         0.487       0.499  0.491
dpsgd_eps1           0.484       0.499  0.489
dpsgd_eps2           0.483       0.499  0.488
dpsgd_eps5           0.482       0.499  0.488
dpsgd_eps10          0.482       0.499  0.488
dpsgd_eps100         0.480       0.499  0.488
tpr@1%fpr
attack          confidence  label_only   loss
model                                        
baseline_e50         0.011         0.0  0.013
baseline_early       0.011         0.0  0.011
dpsgd_eps0.5         0.009         0.0  0.009
dpsgd_eps1           0.008         0.0  0.008
dpsgd_eps2           0.009         0.0  0.009
dpsgd_eps5           0.008         0.0  0.008
dpsgd_eps10          0.009         0.0  0.009
dpsgd_eps100         0.010         0.0  0.010


In [ ]:
# Hidden-attribute test: can an attacker guess gender from the model's output?
from sklearn.ensemble import HistGradientBoostingClassifier
feats = feature_names(); a = feats.index("gender_Male")
def split_attr(X): return np.delete(X, a, axis=1), X[:, a].astype(int)
def fmat(Xo, y, margin=None):
    parts = [Xo, y.reshape(-1, 1)] + ([margin.reshape(-1, 1)] if margin is not None else [])
    return np.hstack(parts)
def fit_att(F, g): return HistGradientBoostingClassifier(random_state=0).fit(F, g)
def auc_of(clf, F, g): return roc_auc_score(g, clf.predict_proba(F)[:, 1])
def margin(m, X):
    z = get_logits(m, X); return z[:, 1] - z[:, 0]

rng = np.random.RandomState(0); n = len(Xte)
mem = rng.choice(len(Xtr), n, replace=False)
Xo_sh, g_sh = split_attr(Xs_tr); Xo_m, g_m = split_attr(Xtr[mem]); Xo_nm, g_nm = split_attr(Xte)
c0 = fit_att(fmat(Xo_sh, ys_tr), g_sh)
base_m, base_nm = auc_of(c0, fmat(Xo_m, ytr[mem]), g_m), auc_of(c0, fmat(Xo_nm, yte), g_nm)
rows = []
for seed in SEEDS:
    for name in NAMES:
        m = load_model(name, seed, input_dim)
        clf = fit_att(fmat(Xo_sh, ys_tr, margin(m, Xs_tr)), g_sh)
        rows.append({"seed": seed, "model": name, "auc_base_member": base_m, "auc_base_nonmember": base_nm,
                     "auc_member": auc_of(clf, fmat(Xo_m, ytr[mem], margin(m, Xtr[mem])), g_m),
                     "auc_nonmember": auc_of(clf, fmat(Xo_nm, yte, margin(m, Xte)), g_nm)})
ai = pd.DataFrame(rows); ai.to_csv(RESULTS + "attribute_inference.csv", index=False)
ai["gain_unseen"] = ai["auc_nonmember"] - ai["auc_base_nonmember"]
ai["memorisation_effect"] = (ai["auc_member"] - ai["auc_base_member"]) - ai["gain_unseen"]
print(ai.groupby("model")[["gain_unseen", "memorisation_effect"]].agg(["mean", "std"]).reindex(NAMES).round(3).to_string())


               gain_unseen        memorisation_effect       
                      mean    std                mean    std
model                                                       
baseline_e50         0.018  0.013              -0.001  0.007
baseline_early      -0.001  0.007              -0.000  0.006
dpsgd_eps0.5         0.174  0.094               0.001  0.009
dpsgd_eps1           0.164  0.101               0.000  0.009
dpsgd_eps2           0.147  0.101              -0.003  0.009
dpsgd_eps5           0.103  0.079               0.001  0.010
dpsgd_eps10          0.062  0.066               0.005  0.008
dpsgd_eps100         0.007  0.012               0.002  0.004
